# Прогноз дохода клиентов финальная версия.


# Импорт

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, gc
from sklearn.model_selection import KFold
from scipy.optimize import minimize_scalar
import warnings; warnings.filterwarnings('ignore')

def wmae(y_true, y_pred, weights): return (weights*np.abs(y_true-y_pred)).mean()
clip = lambda p: np.clip(p, 20000, 1500000)

# Загрузка + устойчивая чистка типов

In [ ]:
NATIVE_CATS = ['gender', 'addrref']
HIGHCARD    = ['dp_ewb_last_employment_position', 'city_smart_name', 'adminarea']
DROP        = ['id', 'target', 'w', 'dt', 'period_last_act_ad']
SALARY = ['salary_6to12m_avg','incomeValue','dp_ils_avg_salary_1y','dp_ils_avg_salary_2y',
 'dp_ils_avg_salary_3y','per_capita_income_rur_amt','salary_median_in_gex_r1',
 'dp_payoutincomedata_payout_avg_3_month','dp_payoutincomedata_payout_max_3_month',
 'dp_payoutincomedata_payout_sum_3_month','dp_payoutincomedata_payout_max_6_month',
 'dp_payoutincomedata_payout_avg_6_month','dp_payoutincomedata_payout_avg_prev_year',
 'profit_income_out_rur_amt_12m','profit_income_out_rur_amt_9m','profit_income_out_rur_amt_l2m',
 'first_salary_income']

train = pd.read_csv('train.csv', decimal=',', sep=';', low_memory=False)
test  = pd.read_csv('test.csv',  decimal=',', sep=';', low_memory=False)
SALARY = [c for c in SALARY if c in train.columns]

str_cols = set(NATIVE_CATS + HIGHCARD + ['id', 'dt'])
def robust_coerce(df):
    for c in df.columns:
        if c in str_cols: continue
        s = df[c]
        if s.dtype == object:
            s = (s.astype(str).str.replace(',', '.', regex=False)
                   .str.replace('None', '', regex=False)
                   .str.replace('nan', '', regex=False).str.strip())
        df[c] = pd.to_numeric(s, errors='coerce').astype('float32')
    return df
train = robust_coerce(train); test = robust_coerce(test)
print(train.shape, test.shape)

(76786, 224) (73214, 222)


# Feature engineering: агрегаты доходных прокси + OOF target-encoding

In [ ]:
def add_salary_aggregates(df):
    sub = df[SALARY]
    agg = pd.DataFrame({'sal_max':sub.max(1),'sal_median':sub.median(1),'sal_mean':sub.mean(1),
                        'sal_std':sub.std(1),'sal_n':sub.notna().sum(1)}, index=df.index)
    logs = pd.DataFrame({'log_'+c: np.log1p(df[c].clip(lower=0)) for c in SALARY}, index=df.index)
    return pd.concat([df, agg.astype('float32'), logs.astype('float32')], axis=1)

train = add_salary_aggregates(train); test = add_salary_aggregates(test); gc.collect()

def _wmap(k, y, w, gm, sm):
    d = pd.DataFrame({'k':np.asarray(k),'y':np.asarray(y),'w':np.asarray(w)})
    return d.groupby('k').apply(lambda g:(np.average(g.y,weights=g.w)*g.w.sum()+gm*sm)/(g.w.sum()+sm))

def te_oof(tr_key, y, w, ns=5, sm=100, seed=42):
    tr_key = pd.Series(np.asarray(tr_key)).astype(str).values
    gm = np.average(y, weights=w); enc = np.full(len(tr_key), gm, 'float32')
    for ti, vi in KFold(ns, shuffle=True, random_state=seed).split(tr_key):
        m = _wmap(tr_key[ti], y[ti], w[ti], gm, sm)
        enc[vi] = pd.Series(tr_key[vi]).map(m).fillna(gm).values
    return enc, _wmap(tr_key, y, w, gm, sm), gm

y_all, w_all = train['target'].values, train['w'].values
for col in HIGHCARD:
    enc, full_map, gm = te_oof(train[col].astype(str), y_all, w_all)
    train['te_'+col] = enc
    test['te_'+col]  = pd.Series(test[col].astype(str).values).map(full_map).fillna(gm).values.astype('float32')

for c in NATIVE_CATS:
    cats_ = train[c].astype(str).astype('category').cat.categories
    train[c] = pd.Categorical(train[c].astype(str), categories=cats_)
    test[c]  = pd.Categorical(test[c].astype(str),  categories=cats_)

FEATURES = [c for c in train.columns if c not in DROP and c not in HIGHCARD]
CATS = list(NATIVE_CATS)
print('features:', len(FEATURES))

features: 241


# Honest holdout: подбираем вес бленда, число итераций
Обучаем на янв–мае, проверяем на июне.

In [ ]:
val_mask = train['dt'].isin(['2024-06-30','30.06.2024']).values; tr_mask = ~val_mask
ytr, wtr = train.loc[tr_mask,'target'].values, train.loc[tr_mask,'w'].values
yv,  wv  = train.loc[val_mask,'target'].values, train.loc[val_mask,'w'].values

P_C = dict(objective='regression_l1', metric='mae', num_leaves=63,  min_child_samples=100,
           learning_rate=0.05, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
           boosting_type='gbdt', verbose=-1, seed=42, n_jobs=-1)
P_D = dict(P_C); P_D.update(num_leaves=127, min_child_samples=40, learning_rate=0.03, feature_fraction=0.7)
P_Q = dict(P_C); P_Q.update(objective='quantile', alpha=0.6); P_Q.pop('metric')

def fit_es(par, nb=4000):
    dtr = lgb.Dataset(train.loc[tr_mask,FEATURES], ytr, weight=wtr, categorical_feature=CATS)
    dva = lgb.Dataset(train.loc[val_mask,FEATURES], yv, weight=wv, categorical_feature=CATS, reference=dtr)
    m = lgb.train(par, dtr, num_boost_round=nb, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(120, verbose=False)])
    return m, clip(m.predict(train.loc[val_mask,FEATURES], num_iteration=m.best_iteration))

mC, pC = fit_es(P_C); mD, pD = fit_es(P_D); mQ, pQ = fit_es(P_Q)
r = minimize_scalar(lambda b: wmae(yv, clip(b*pD+(1-b)*pC), wv), bounds=(0,1), method='bounded')
W_DEEP = r.x
print(f'C WMAE={wmae(yv,pC,wv):.2f} | deep WMAE={wmae(yv,pD,wv):.2f} | W_DEEP={W_DEEP:.3f}')
print(f'holdout blend WMAE={wmae(yv, clip(W_DEEP*pD+(1-W_DEEP)*pC), wv):.2f} (справочно; на тесте хвост тяжелее)')
N_C, N_D, N_Q = int(mC.best_iteration*1.05), int(mD.best_iteration*1.05), int(mQ.best_iteration*1.05)

C WMAE=33786.22 | deep WMAE=34027.94 | W_DEEP=0.382
holdout blend WMAE=33681.32 (справочно; на тесте хвост тяжелее)


# Финальное обучение: 5 фолдов × 3 сида, предсказание теста


In [ ]:
SEEDS = [42, 777, 2024]
def train_bagged(par, nrounds):
    preds = np.zeros(len(test)); k = 0
    kf = KFold(n_splits=5, shuffle=True, random_state=par['seed'])
    for s in SEEDS:
        p = dict(par); p['seed'] = s; p['bagging_seed'] = s; p['feature_fraction_seed'] = s
        for ti, _ in kf.split(train):
            d = lgb.Dataset(train.iloc[ti][FEATURES], y_all[ti], weight=w_all[ti], categorical_feature=CATS)
            m = lgb.train(p, d, num_boost_round=nrounds)
            preds += clip(m.predict(test[FEATURES])); k += 1
    return preds / k

tC = train_bagged(P_C, N_C); print('C готов')
tD = train_bagged(P_D, N_D); print('deep готов')
tQ = train_bagged(P_Q, N_Q); print('quantile готов')

C готов
deep готов
quantile готов


# Финальный прогноз


In [ ]:
TAIL_T = 80000
TAIL_A = 0.7
# --- калибровка прямой зарплаты на всём train (для оверрайда) ---
import numpy as _np
_sal_all = pd.to_numeric(train['salary_6to12m_avg'], errors='coerce').values
_m = ~_np.isnan(_sal_all)
from scipy.optimize import minimize_scalar as _ms
K_SAL = _ms(lambda k: (w_all[_m]*_np.abs(y_all[_m]-clip(k*_sal_all[_m]))).mean(), bounds=(0.2,3), method='bounded').x
A_SAL = 0.65  # доля доверия зарплате для строк, где она есть

blend = clip(W_DEEP*tD + (1-W_DEEP)*tC)
out = blend.copy(); hi = out > TAIL_T
out[hi] = TAIL_A*tQ[hi] + (1-TAIL_A)*out[hi]; out = clip(out)

# ОВЕРРАЙД: где есть прямая зарплата (~0.935 корр.), доверяем ей, а не разбавленному прогнозу
_sal_te = pd.to_numeric(test['salary_6to12m_avg'], errors='coerce').values
_pres = ~_np.isnan(_sal_te); _psal = clip(K_SAL*_sal_te)
out[_pres] = clip(A_SAL*_psal[_pres] + (1-A_SAL)*out[_pres])
test_preds = clip(out)
print(f'salary override: k={K_SAL:.3f}, переопределено {int(_pres.sum())} строк ({_pres.mean():.1%})')

test['predict'] = test_preds
test[['id','predict']].set_index('id').to_csv('commit.csv', decimal=',', sep=';')
print('commit.csv сохранён |', f'mean={test_preds.mean():.0f} p90={_np.percentile(test_preds,90):.0f}')


salary override: k=0.929, переопределено 7634 строк (10.4%)
commit.csv сохранён | mean=101071 p90=205252
